# Живые визуализации своего обученного pipeline

Этот notebook читает pipeline и контрольные данные из локального run.
Сначала выполните `python examples/train_visual_demo.py` из корня проекта
или задайте `RUN_DIRECTORY` для своего совместимого запуска ниже.

Для готовой модели откройте `06_trained_interactive_demo.ipynb`:
он загрузит отдельные файлы весов из `demo/pipeline` в репозитории.
Пакет устанавливается из GitHub; нужны Python 3.11+, Git и интернет.


In [ ]:
# Установить пакет из GitHub в Python текущего Jupyter kernel.
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "diffusion-lab[interactive] @ git+" + REPOSITORY_URL,
])


In [ ]:
from pathlib import Path
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/diffusion_lab").is_dir()), Path.cwd())
RUN_DIRECTORY = ROOT / "outputs/visual_demo_rays"  # замените на свой run
PIPELINE_PATH = RUN_DIRECTORY / "pipeline"
REFERENCE_PATH = RUN_DIRECTORY / "reference.npz"
if not PIPELINE_PATH.exists():
    raise FileNotFoundError("Run examples/train_visual_demo.py first, or use 06_trained_interactive_demo.ipynb")


In [ ]:
import numpy as np
import torch
from IPython.display import display
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
from diffusion_lab import ResearchDiffusionPipeline
from diffusion_lab.environments import SampleDataset
from diffusion_lab.visualization import notebook_field_explorer, notebook_sampling_explorer

torch.set_num_threads(4)
pipeline = ResearchDiffusionPipeline.from_pretrained(
    PIPELINE_PATH, local_files_only=True, low_cpu_mem_usage=False,
).to("cpu")
with np.load(REFERENCE_PATH, allow_pickle=False) as reference:
    dataset = SampleDataset(reference["samples"], reference["conditions"], num_classes=4,
                            metadata={"name": "rotating_rays", "seed": 124})
print(type(pipeline.model).__name__, type(pipeline.scheduler).__name__)


## Поле: condition, timestep, CFG, potential

Слайдеры пересчитывают поле по модели при отпускании ползунка. Цвет стрелок
кодирует исходную величину; нормировка меняет только отображение.
`Recover potential` решает F≈−∇U по исходным векторам и показывает residual/curl.
Ненулевой curl означает, что восстановленная U является проекцией.


In [ ]:
fields = notebook_field_explorer(pipeline, dataset, resolution=32,
                                 timestep=349, guidance_scale=1.5)
fields.controls["condition"].value = 1  # Class 0; 0 означает Unconditional
fields.controls["potential"].value = True
display(fields)


## Генерация и реальные кадры денойзинга

Выберите condition / CFG / steps / seed и нажмите **Generate**. Play и Frame
проигрывают сохранённые состояния этого же запуска; перемещение Frame не
генерирует новый шум. Контролы не запускают обучение в фоне.


In [ ]:
sampling = notebook_sampling_explorer(pipeline, dataset, num_samples=256,
                                       steps=100, seed=127, guidance_scale=1.5)
sampling.generate()
display(sampling)


## Использовать контролы в своём исследовании

После обычного PyTorch обучения передайте свой `ResearchDiffusionPipeline`:
```python
fields = notebook_field_explorer(pipeline, dataset)
sampling = notebook_sampling_explorer(pipeline, dataset)
display(fields, sampling)
```
Виджеты доступны через `.controls`, данные последнего расчёта — `.state`.
Для continuous conditioning можно передать `condition_options={"name": tensor}`.
1D/2D/3D поля поддерживаются для модели точек; у модели траекторий используйте
sampling explorer с отображением стен и концов пути. Свою модель, optimizer и
loss меняйте обычным PyTorch кодом из `03` / `04`.
В Colab при необходимости включите manager стандартных widgets в окружении;
в обычном Jupyter зависимости `[interactive]` включают ipywidgets.
